# 027 — swB: save a custom NVM config from python + frequency-move test

Bench smoke test, deliberately bare: **no logging setup, no Bokeh monitor, no telemetry sink.**
Switch only — the PSUs are never touched here.

**Plan** (2026-07-07):
1. Connect swB, load config **89** (= file `[Configuration90]`, SwitchSym 1 MHz baseline).
2. Diagnostic for the OPEN readback quirk: poll `get_pulser_delay(0)` after the load — NVM stores delay **7** for this config, the SW plugin read **0**.
3. Tweak the working set to **2 kHz / 50 %** (campaign-style width re-fit), **save to python slot 119** (= file `[Configuration120]`, empty in the cfg export) and name it `TestTestTest2kHz`.
4. Enumerate the NVM — if slot 119 shows up with that name, python-made configs work.
5. Round-trip proof: standby, reload slot 119, read back 2 kHz.
6. Section 2: simple change-frequency ladder on the working set.

Numbering reminder: python index N = file `[Configuration N+1]`; python 0 = Standby.
NVM writes signal busy via **CTS** — the sleeps after save/name are not decoration.
Scope: whatever your current hookup shows (Mon0/Mon1 need PSU voltage; DIO0 = PulsOut0 is a logic signal).

In [ ]:
"""Initialize swB. SIM=True exercises the sim layer instead (no COM port) —
but the NVM enumeration/name cells are real-hardware-only (raw DLL exports)."""
import sys
import time
import tomllib
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
from devices.cgc import SW

SIM = False  # True = simulated device; run with False at the bench


def com_from_lab_config(key="swB", fallback=10):
    """COM number from the canonical lab_config.toml; fallback = confirmed table 2026-07-06."""
    p = Path.cwd().parents[2] / "lab_services" / "lab_config.toml"
    try:
        return tomllib.loads(p.read_text(encoding="utf-8"))["com_ports"][key]
    except Exception:
        return fallback


COM_SWB = com_from_lab_config()
swB = SW("swB", com=COM_SWB, port=0, test_mode=SIM, skip_sensors=(0,))
assert swB.connect(), "connect failed — is the Explorer SW plugin (or another notebook) holding the port?"
print(f"swB connected on COM{COM_SWB} (SIM={SIM})")

## 1. Config bring-up + readback diagnostic

Loads the campaign baseline, then polls the pulser-delay readback. `[Configuration90]` stores
`Pulser0=7,48` (delay register 7, width 48). If the readback shows 7 only after a delay →
settle-time issue; if it never shows 7 → the readback does not serve the loaded working set
(either way: closes the OPEN quirk on the cgc-sw KB page).

In [ ]:
# CHANGE swB: load NVM config 89 ([Configuration90] = 1MHz->SwitchSym+DIO0,Osc->DIO1)
print("load_current_config(89) ->", swB.load_current_config(89))

status, period = swB.get_oscillator_period()
print(f"period register {period} -> f = {swB.CLOCK / (period + swB.OSC_OFFSET) / 1e3:.3f} kHz (expect 1000.000)")

# readback-quirk diagnostic: cumulative waits, delay + width after each
waited = 0.0
for wait in (0.1, 0.4, 0.5, 1.0, 3.0):
    time.sleep(wait)
    waited += wait
    print(f"+{waited:>4.1f} s   get_pulser_delay(0) -> {swB.get_pulser_delay(0)}   "
          f"get_pulser_width(0) -> {swB.get_pulser_width(0)}   (NVM stores delay 7, width 48)")

## 2. Tweak the working set to 2 kHz / 50 % and save it as a new NVM config

The move uses the campaign order (width→1 first so width ≥ period can never happen
mid-move, then the period, then the duty re-fit) — same as notebook 025's
`swb_set_frequency`, **no readback validation**, statuses printed instead.

In [ ]:
def set_rf_raw(sw, f_khz, duty=0.5, pulser=0):
    """Campaign frequency move (025 swb_set_frequency): width -> 1, period, duty re-fit.
    Returns the three statuses (0 = OK) — no readback validation."""
    with sw.thread_lock:
        s1 = sw.set_pulser_width(pulser, 1)   # CHANGE swB: pulser width -> minimum
        s2 = sw.set_frequency_khz(f_khz)      # CHANGE swB: oscillator -> f_khz
        s3 = sw.set_duty_cycle(pulser, duty)  # CHANGE swB: width -> duty of the new period
    return s1, s2, s3


# CHANGE swB: working set -> 2 kHz / 50 % (pulser 0)
print("set_rf_raw(2 kHz, 50 %) ->", set_rf_raw(swB, 2, 0.5))

status, period = swB.get_oscillator_period()
status_w, width = swB.get_pulser_width(0)
print(f"period {period} (expect {round(swB.CLOCK / 2e3 - swB.OSC_OFFSET)}), "
      f"width {width} (expect {round(0.5 * (period + swB.OSC_OFFSET) - swB.PULSER_WIDTH_OFFSET)})")

# CHANGE swB: save the working set -> NVM slot python 119 ([Configuration120], empty in the cfg export)
print("save_current_config(119) ->", swB.save_current_config(119))
time.sleep(2)  # NVM write, CTS busy — do not talk to the device during the save

if SIM:
    print("set_config_name is a raw DLL export — real hardware only, skipped in SIM")
else:
    # CHANGE swB: name NVM slot 119
    print("set_config_name(119, 'TestTestTest2kHz') ->", swB.set_config_name(119, "TestTestTest2kHz"))
    time.sleep(1)  # name is an NVM write too

## 3. Verify: enumerate the NVM — slot 119 must be there with the test name

In [ ]:
if SIM:
    print("NVM enumeration is real-hardware only — run with SIM=False at the bench")
else:
    status, active, valid = swB.get_config_list()
    valid_slots = [i for i, v in enumerate(valid) if v]
    print(f"get_config_list -> status {status}, {len(valid_slots)} valid slots")
    print("valid python slots:", valid_slots)
    name_status, name = swB.get_config_name(119)
    print(f"slot 119 (file [Configuration120]) name -> {name!r}")
    if 119 in valid_slots and name == "TestTestTest2kHz":
        print("PASS — python-made NVM configs work")
    else:
        print("CHECK FAILED — slot missing or name mismatch")

## 4. Round-trip proof: standby → reload slot 119 → read back 2 kHz

In [ ]:
# CHANGE swB: park in standby (python 0), then reload the saved slot
print("standby ->", swB.load_current_config(0))
time.sleep(0.5)
print("load 119 ->", swB.load_current_config(119))
time.sleep(0.5)

status, period = swB.get_oscillator_period()
status_w, width = swB.get_pulser_width(0)
f_khz = swB.CLOCK / (period + swB.OSC_OFFSET) / 1e3
duty = (width + swB.PULSER_WIDTH_OFFSET) / (period + swB.OSC_OFFSET)
print(f"readback: f = {f_khz:.3f} kHz (expect 2.000), duty = {duty * 100:.2f} % (expect 50.00)")

## 5. Section 2 — simple change-frequency test

Ladder on the live working set, duty re-fit at every step (watch the scope between steps).
This is the exact write sequence the campaign verified to 1 MHz — if the statuses are 0 and
the readback matches here, the Explorer plugin's `set_rf` path can come back (with the
delay-readback validation removed or demoted, pending the cell-1 diagnostic).

In [ ]:
# CHANGE swB: frequency ladder 2 -> 100 kHz at 50 % duty (pulser 0)
for f in (2, 5, 10, 50, 100):
    statuses = set_rf_raw(swB, f, 0.5)
    status, period = swB.get_oscillator_period()
    status_w, width = swB.get_pulser_width(0)
    f_read = swB.CLOCK / (period + swB.OSC_OFFSET) / 1e3
    duty = (width + swB.PULSER_WIDTH_OFFSET) / (period + swB.OSC_OFFSET)
    print(f"{f:>4} kHz: statuses {statuses}   readback f = {f_read:8.3f} kHz   duty = {duty * 100:6.2f} %")
    time.sleep(1)

## 6. Teardown — always run this cell before closing the kernel

In [ ]:
# CHANGE swB: park in standby + release the COM port
print("standby ->", swB.load_current_config(0))
swB.disconnect()
print("disconnected")